# 04 — Particle Swarm Optimization (PSO) Feature Selection

Binary PSO searches for a compact feature mask. This notebook uses the shared experiment pipeline so all optimizers receive the same data splits, classifiers, seeds, and evaluation rules.


In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "utils").exists():
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT))

import random
import numpy as np


## Binary Particle Swarm Optimization

In [2]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def run_bpso(
    obj_func,
    n_features,
    pop_size=30,
    iterations=50,
    w=0.4,
    c1=2.05,
    c2=2.05,
):
    # Initialize particles
    positions = np.random.randint(0, 2, (pop_size, n_features))
    velocities = np.random.uniform(-1, 1, (pop_size, n_features))

    # Personal best
    pbest_positions = positions.copy()
    pbest_scores = np.array([obj_func(p) for p in positions])

    # Global best
    best_idx = np.argmin(pbest_scores)
    gbest_position = pbest_positions[best_idx].copy()
    gbest_score = pbest_scores[best_idx]

    convergence = []

    for _ in range(iterations):

        for i in range(pop_size):

            r1 = np.random.rand(n_features)
            r2 = np.random.rand(n_features)

            velocities[i] = (
                w * velocities[i]
                + c1 * r1 * (pbest_positions[i] - positions[i])
                + c2 * r2 * (gbest_position - positions[i])
            )

            probs = sigmoid(velocities[i])

            positions[i] = (np.random.rand(n_features) < probs).astype(int)

            # Prevent empty feature subset
            if positions[i].sum() == 0:
                positions[i, np.random.randint(n_features)] = 1

            score = obj_func(positions[i])

            if score < pbest_scores[i]:
                pbest_scores[i] = score
                pbest_positions[i] = positions[i].copy()

                if score < gbest_score:
                    gbest_score = score
                    gbest_position = positions[i].copy()

        convergence.append(gbest_score)

    return gbest_position, gbest_score, convergence

## Run the complete feature-selection experiment

The shared experiment runner supplies the configured datasets, classifiers,
optimizer seeds, population size, and iteration count. Feature selection uses
the validation set. The test set is evaluated only after the final mask has
been selected.


In [3]:
from utils.experiments import run_feature_selector


def pso_runner(
    objective,
    n_features,
    pop_size,
    iterations,
):
    return run_bpso(
        obj_func=objective,
        n_features=n_features,
        pop_size=pop_size,
        iterations=iterations,
        w=0.4,
        c1=2.05,
        c2=2.05,
    )


pso_results = run_feature_selector("PSO", pso_runner)
pso_results.tail()


Saved: ('breast', 'svm', 0)


Saved: ('breast', 'svm', 1)


Saved: ('breast', 'svm', 2)


Saved: ('breast', 'svm', 3)


Saved: ('breast', 'svm', 4)


...

Saved: ('heart', 'xgboost', 1)


Saved: ('heart', 'xgboost', 2)


Saved: ('heart', 'xgboost', 3)


Saved: ('heart', 'xgboost', 4)


,Dataset,Classifier,Algorithm,Seed,ValidationFitness,Accuracy,Precision,Recall,F1,ROC_AUC,Features,SelectionRuntime,TestRuntime,SelectedFeatureNames,MaskFile,ConvergenceFile
25,heart,xgboost,PSO,0,0.123570,0.788043,0.831579,0.774510,0.802030,0.880560,13,19.999352,0.090068,"[""chol"", ""thalch"", ""oldpeak"", ""ca"", ""cp_asympt...",results/final/artifacts/heart__xgboost__pso__s...,results/final/artifacts/heart__xgboost__pso__s...
26,heart,xgboost,PSO,1,0.125570,0.788043,0.831579,0.774510,0.802030,0.893950,18,20.147956,0.091120,"[""age"", ""chol"", ""oldpeak"", ""ca"", ""sex_Male"", ""...",results/final/artifacts/heart__xgboost__pso__s...,results/final/artifacts/heart__xgboost__pso__s...
27,heart,xgboost,PSO,2,0.124370,0.793478,0.840426,0.774510,0.806122,0.880201,15,20.629725,0.093863,"[""chol"", ""thalch"", ""oldpeak"", ""ca"", ""cp_asympt...",results/final/artifacts/heart__xgboost__pso__s...,results/final/artifacts/heart__xgboost__pso__s...
28,heart,xgboost,PSO,3,0.118589,0.793478,0.855556,0.754902,0.802083,0.892276,14,20.958563,0.095245,"[""chol"", ""thalch"", ""oldpeak"", ""ca"", ""sex_Male""...",results/final/artifacts/heart__xgboost__pso__s...,results/final/artifacts/heart__xgboost__pso__s...
29,heart,xgboost,PSO,4,0.123970,0.771739,0.826087,0.745098,0.783505,0.886896,14,21.058447,0.097097,"[""chol"", ""thalch"", ""oldpeak"", ""sex_Female"", ""s...",results/final/artifacts/heart__xgboost__pso__s...,results/final/artifacts/heart__xgboost__pso__s...
